# TripoSR on Colab
画像1枚 → 3Dメッシュ（.glb）を無料GPUで生成するノートブック。

使い方：上から順に全部のセルを実行 → 画像アップロード → 最後に .glb をダウンロード。
- TripoSR: MIT license (VAST-AI-Research/TripoSR), weights: stabilityai/TripoSR
- 無料枠の T4 GPU で十分回る（VRAM 6GB級でOK）

In [ ]:
# 1) セットアップ（成功実績のある版：torchmcubes をビルドせず PyMCubes に差し替え）
%cd /content
!rm -rf TripoSR
!git clone -q https://github.com/VAST-AI-Research/TripoSR.git
%cd TripoSR

# 競合しやすいパッケージを除外して安全にインストール（torch 等は Colab 標準を使う）
!grep -vE "torch|torchvision|Pillow|trimesh|onnxruntime|torchmcubes|xatlas" requirements.txt > req_clean.txt
!pip install -q -r req_clean.txt

# コンパイル不要な代替 Marching Cubes と xatlas を導入
!pip install -q PyMCubes xatlas trimesh onnxruntime

# tsr/models/isosurface.py の torchmcubes を PyMCubes ラッパーに差し替え
isosurface_path = "tsr/models/isosurface.py"
with open(isosurface_path) as f:
    code = f.read()
patch_code = """
import mcubes
class MockTorchMCubes:
    @staticmethod
    def marching_cubes(pdf, isovalue):
        device = pdf.device
        pdf_np = pdf.detach().cpu().numpy()
        verts, faces = mcubes.marching_cubes(pdf_np, isovalue)
        return torch.from_numpy(verts.copy()).to(device), torch.from_numpy(faces.astype("int32").copy()).to(device)
torchmcubes = MockTorchMCubes()
"""
code = code.replace("import torchmcubes", patch_code)
with open(isosurface_path, "w") as f:
    f.write(code)
print("setup done, patched torchmcubes with PyMCubes successfully!")

# transformers 新版の DINOv2 キー名変更に対応：ckpt のキーを旧→新へリマップ
_system_path = "tsr/system.py"
_s = open(_system_path).read()
_header = """
def _remap_ckpt(ckpt):
    import re as _re
    def _f(k):
        m = _re.match(r"image_tokenizer\\.model\\.encoder\\.layer\\.(\\d+)\\.(.*)", k)
        if not m:
            return k
        i, rest = m.groups()
        rest = (rest.replace("attention.attention.query", "attention.q_proj")
                    .replace("attention.attention.key", "attention.k_proj")
                    .replace("attention.attention.value", "attention.v_proj")
                    .replace("attention.output.dense", "attention.o_proj")
                    .replace("intermediate.dense", "mlp.fc1")
                    .replace("output.dense", "mlp.fc2"))
        return "image_tokenizer.model.layers.{}.{}".format(i, rest)
    return {_f(k): v for k, v in ckpt.items()}

"""
if "_remap_ckpt" not in _s:
    assert "model.load_state_dict(ckpt)" in _s, "system.py のロード行が想定と違います（パッチ中止）"
    _s = _header + _s.replace("model.load_state_dict(ckpt)", "model.load_state_dict(_remap_ckpt(ckpt))")
    open(_system_path, "w").write(_s)
print("patched system.py with _remap_ckpt")


In [ ]:
# 2) 画像をアップロード
from google.colab import files
up = files.upload()
img_path = list(up.keys())[0]
print(img_path)

In [ ]:
# 3) 前処理：三面図なら FRONT だけ切り抜く（単体画像なら CROP=None のまま）
from PIL import Image
CROP = (0.02, 0.06, 0.34, 0.97)  # (left, top, right, bottom) 0..1 / None で無効
im = Image.open(img_path).convert('RGB')
if CROP:
    w, h = im.size
    im = im.crop((int(CROP[0]*w), int(CROP[1]*h), int(CROP[2]*w), int(CROP[3]*h)))
im.save('input.png')
im

In [ ]:
# 4) 推論（重みは初回だけ自動ダウンロード、約1.6GB）
!python run.py input.png --output-dir out
import glob
print(glob.glob('out/**/*.obj', recursive=True))

In [ ]:
# 5) .glb に変換してダウンロード
import glob, trimesh
obj = sorted(glob.glob('out/**/*.obj', recursive=True))[0]
scene = trimesh.load(obj)
scene.export('mesh.glb')
from google.colab import files
files.download('mesh.glb')
print('downloaded mesh.glb')

## メモ
- 無料枠はセッションが切れると全部消える。生成したらすぐダウンロードすること
- 顔や刺繍などの細部は潰れがち。背面はAIの補完（捏造）
- 仕上げは Blender 側（krimskrams/models の build_model.py 流儀）で行う